# Lab 6 - Give an agent tools

## What will you do?

On Day 1 you gave an agent a knowledge base and it answered questions about WHO guidelines. That tool only ever **read**. This lab adds tools that reach into your own systems, and then tools that **change** something - which is where the engineering gets serious, because a wrong answer misinforms one person and a wrong action is still there tomorrow.

A **tool** is a capability you attach to an agent: a description of what it does and how to call it. The agent decides when to use it. You never write `if user_asks_about_x: call_y()`, and that declarative style is the part people find hardest - your job is to describe the tool clearly enough that the model can work out when it applies.

Tools differ in one way that matters more than any other: **who actually runs the code.**

| | Who runs it | Example | You must |
|---|---|---|---|
| Service-run | Foundry, in Azure | Code interpreter, file search | Attach it |
| Remote (MCP) | Someone else's server | The knowledge base from Lab 3 | Attach it, and authorise it |
| **Function** | **Your own process** | The two you write today | **Attach it, run it, and return the result** |

Function calling is the only tool Foundry cannot put in a shared toolbox, and the reason is exactly this: *client-side execution*. The service can describe your function, but it can never call it. That round trip is the core of this lab.

In this lab you will:

1. Look at a local system the model has never seen.
2. Declare it as a function tool.
3. Complete the call-and-return round trip by hand.
4. Combine it with the Lab 3 knowledge base to answer a question neither tool can answer alone.
5. Add an action that changes data, and put a human in front of it.

### First, the objection

For a question like *"what is Ward 4B's staffing score?"* an agent with a tool is a **bad** engineering choice. A dashboard is faster, cheaper, deterministic and auditable. If that is your use case, build the dashboard.

A tool-using agent earns its cost in three situations, and this lab is built to show them:

1. **The join is the work.** A dashboard shows `staffing: 1/5`. It cannot tell you which WHO recommendation that breaches, quote it, and cite it - that lives in a 91-page PDF in a different system with no shared schema. Today a human does that join by hand.
2. **Nobody built that screen.** A UI answers questions someone anticipated and shipped. New question, new sprint.
3. **Two systems, one answer.** Joining a JSON store and a guideline document in a UI is a project.

You will run a trivial question first and see that a table would have done it better. Then you will run one that no table can answer.

> **This is a workshop exercise.** The WHO guidance is real. The hospital, its wards and its scores are invented for this lab.

## New words

- **Tool** - a capability attached to an agent: a name, a description, and a parameter schema. The model chooses when to call it.
- **Function tool** - a tool whose code runs in *your* process. The service asks; you execute; you return the result.
- **Tool call** - the model's request to use a tool, carrying arguments it generated.
- **Round trip** - request, execute, return, answer. A function tool needs two calls to the service, not one.
- **`call_id`** - the identifier tying a result back to the request that asked for it.
- **Strict schema** - `strict: true`, which forces arguments to match your JSON schema exactly.
- **Approval gate** - a human decision point before a tool with consequences runs.
- **System of record** - the place a fact actually lives. For today's data that is a JSON file; in production it is a database or an API.

## Before you start

- Python 3.11 or later, with a notebook kernel selected, and `az login` completed.
- Labs 1 to 3 finished. This lab reuses that project, model deployment, knowledge base and connection.
- The file `labs/data/hospital/ipc_self_assessment.json`, which ships with this repository.

**A name collision worth knowing about.** The older `azure-ai-agents` package also had a class called `FunctionTool`, and *that* one accepted Python callables and ran them for you. The `FunctionTool` in `azure-ai-projects` is a **schema declaration only**. It holds no reference to your function and will never call it. If you have seen sample code passing functions into `FunctionTool(...)` or using `ToolSet` and `enable_auto_function_calls`, that is the other library.

Microsoft Agent Framework - the library from Lab 4 - *does* run functions for you, via `@tool`. This lab does the round trip by hand first, because a framework that hides the loop is much easier to debug once you have seen the loop.

**How the To-Do sections work.** Replace each `...` blank and run the cell with **Shift+Enter**. A blank left open stops the cell and names it.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect

Fill in the five settings, or set them as environment variables before starting the kernel. They are the same ones Lab 3 used. Your instructor provides them, and none of them is a secret.

| Setting | What it is |
|---|---|
| `AZURE_AI_PROJECT_ENDPOINT` | Your Foundry project |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | The model your agent runs on, for example `gpt-5.4-mini` |
| `AZURE_SEARCH_ENDPOINT` | The Search service holding the knowledge base |
| `AZURE_SEARCH_KNOWLEDGE_BASE` | The knowledge base from Lab 3 |
| `AZURE_KB_CONNECTION_NAME` | The project connection that authenticates to it |

**Run the cell. You should see** `Ready.` and a random suffix. No agent exists yet.

In [ ]:
import json
import os
import sys
from pathlib import Path
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FunctionTool, MCPTool, PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your nonsecret settings. Paste them between the quotes, or set them as
# environment variables before starting the kernel.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "")
KNOWLEDGE_BASE = os.getenv("AZURE_SEARCH_KNOWLEDGE_BASE", "")
KB_CONNECTION_NAME = os.getenv("AZURE_KB_CONNECTION_NAME", "")

SEARCH_API_VERSION = "2026-08-01-preview"
KB_MCP_URL = f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE}/mcp?api-version={SEARCH_API_VERSION}"

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
        "AZURE_SEARCH_ENDPOINT": SEARCH_ENDPOINT,
        "AZURE_SEARCH_KNOWLEDGE_BASE": KNOWLEDGE_BASE,
        "AZURE_KB_CONNECTION_NAME": KB_CONNECTION_NAME,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


SUFFIX = uuid4().hex[:8]

credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)

try:
    deployments = [d.name for d in project.deployments.list()]
except Exception:  # listing needs a role you may not have; skip the check if so.
    deployments = []
if deployments and MODEL_DEPLOYMENT not in deployments:
    raise ValueError(
        f"This project has no deployment named {MODEL_DEPLOYMENT!r}. "
        f"Available: {', '.join(deployments)}."
    )

print(f"Ready. Suffix: {SUFFIX}")

## 1. The system the model has never seen

The WHO infection prevention and control guideline - already in your knowledge base - defines **eight core components** every facility needs. Your hospital scores each ward against them.

That self-assessment is the kind of data no model can ever know. It did not exist when the model was trained, it changes weekly, and it is specific to one organisation. There is no prompt you can write that produces it and no amount of model quality that helps. It has to be fetched.

The next cell loads the file and defines two ordinary Python functions. Nothing about them is AI-specific yet - they are just the code you would write anyway to read your own data.

| Function | Reads or writes | Consequence if it goes wrong |
|---|---|---|
| `get_ipc_assessment(ward)` | Reads | A wrong answer |
| `schedule_ipc_review(ward, component, reason)` | **Writes** | A meeting in someone's calendar |

That difference drives section 5.

**Run the cell. You should see** the three wards and, for Ward 4B, its two weakest components.

In [ ]:
DATA_FILE = Path("../data/hospital/ipc_self_assessment.json")
if not DATA_FILE.exists():  # tolerate a different working directory
    DATA_FILE = Path("labs/data/hospital/ipc_self_assessment.json")

ASSESSMENT = json.loads(DATA_FILE.read_text())
COMPONENTS = ASSESSMENT["components"]
WARDS = {ward["ward"]: ward for ward in ASSESSMENT["wards"]}

# Every review this notebook books ends up here. It starts empty on purpose.
SCHEDULED_REVIEWS = []


def get_ipc_assessment(ward: str) -> dict:
    """Return one ward's IPC self-assessment scores. Read-only."""
    record = WARDS.get(ward.strip().upper())
    if record is None:
        return {"error": f"No ward {ward!r}. Known wards: {', '.join(WARDS)}."}
    return {
        "ward": record["ward"],
        "name": record["name"],
        "beds": record["beds"],
        "last_assessed": record["last_assessed"],
        "scale": ASSESSMENT["scale"]["meaning"],
        "scores": {COMPONENTS[key]: value for key, value in record["scores"].items()},
        "notes": record["notes"],
    }


def schedule_ipc_review(ward: str, component: str, reason: str) -> dict:
    """Book an internal IPC review. This changes data."""
    record = WARDS.get(ward.strip().upper())
    if record is None:
        return {"error": f"No ward {ward!r}. Known wards: {', '.join(WARDS)}."}
    review = {"ward": record["ward"], "component": component, "reason": reason}
    SCHEDULED_REVIEWS.append(review)
    return {"status": "scheduled", "review_id": f"IPC-{len(SCHEDULED_REVIEWS):03d}", **review}


print(f"Loaded {DATA_FILE.name}: {len(WARDS)} wards, {len(COMPONENTS)} WHO core components.\n")
for ward in WARDS.values():
    weakest = sorted(ward["scores"].items(), key=lambda item: item[1])[:2]
    summary = ", ".join(f"{COMPONENTS[key]} = {value}/5" for key, value in weakest)
    print(f"  Ward {ward['ward']:<4} {ward['name']:<16} weakest: {summary}")

## 2. Declare the function as a tool

The agent needs a description of your function, not the function itself. Four fields:

| Field | Purpose |
|---|---|
| `name` | What the model calls |
| `description` | **When** to call it. This is the field that decides whether the tool ever gets used |
| `parameters` | A JSON Schema for the arguments |
| `strict` | With `true`, arguments must match the schema exactly |

`description` does more work than people expect. It is not documentation for you, it is the model's only basis for deciding whether this tool is relevant to a question. A tool that is never called and a tool that is broken look identical from the outside, and the usual cause is the same: a description too vague for the model to tell that the tool applied.

For `strict: true` the schema also needs `"additionalProperties": false` and every property listed in `required`.

### To-Do 1 - Describe the arguments

**Goal:** a schema tight enough that the model can only send a ward identifier.

**Steps**

1. Write `PARAMETERS` as a JSON Schema object with one string property, `ward`.
2. Give that property a `description` naming the valid wards. The model reads it.
3. Mark it required and set `additionalProperties` to `false`.

**Predict:** if you left the `description` off `ward`, what would the model send for *"how is general medicine doing?"*

**Run the cell. You should see** the tool summarised, with its one parameter.

<details><summary>Hint</summary>

Standard JSON Schema: `{"type": "object", "properties": {...}, "required": [...], "additionalProperties": False}`. Listing the ward identifiers in the description is what lets the model map "general medicine" to `4B`.

</details>

<details><summary>Show solution code</summary>

```python
PARAMETERS = {
    "type": "object",
    "properties": {
        "ward": {
            "type": "string",
            "description": "Ward identifier, one of: 4B (General Medicine), 2A (Surgical), ICU (Intensive Care).",
        }
    },
    "required": ["ward"],
    "additionalProperties": False,
}
```

</details>

In [ ]:
PARAMETERS = ...  # TODO 1: a JSON Schema object with one required string property, `ward`.
check_todos(PARAMETERS=PARAMETERS)

assessment_tool = FunctionTool(
    type="function",
    name="get_ipc_assessment",
    description=(
        "Look up one hospital ward's internal infection prevention and control (IPC) "
        "self-assessment scores against the WHO core components. Use this for any question "
        "about how a ward is actually performing. This data is internal and cannot be guessed."
    ),
    parameters=PARAMETERS,
    strict=True,
)

print(f"Tool: {assessment_tool.name}")
print(f"  parameters: {', '.join(PARAMETERS['properties'])}")
print(f"  required  : {', '.join(PARAMETERS['required'])}")
print(f"  strict    : {assessment_tool.strict}")

## 3. The round trip

A service-run tool is one call in, one answer out. A function tool is not, because the service has to stop and wait for you:

```text
you  ->  "how is Ward 4B doing?"
         service: I need get_ipc_assessment(ward="4B")     <- function_call item
you  ->  run it yourself, send the result                  <- function_call_output item
         service: "Ward 4B scores 1 of 5 on staffing..."   <- message item
```

Five steps: define the tool, create the agent, send a prompt, **execute and return**, get the final response. Step four is yours alone. The service will wait, and it will wait forever, because nothing else can run your code.

Three details decide whether this works:

- The first response's `status` is already `completed`. It is not "waiting" - a `function_call` is simply an item in `response.output`, and you have to look for it.
- `arguments` is a **JSON string**, not a dict. Parse it.
- `output` must also be a **string**. Serialise your result.
- `call_id` ties your result to the request. With several calls in one response, this is the only thing that matches them up.

The loop below repeats until no more calls come back, because a model may need a second lookup after seeing the first result.

### To-Do 2 - Return the result

**Goal:** a finished answer that used your data.

**Steps**

1. Set `CALL_ID` to the identifier the service uses to match a result to its request.
2. Set `OUTPUT_JSON` to the function's result, encoded as a JSON string.
3. Run the cell and read the transcript.

**Predict:** the question below is a plain lookup. Would a dashboard have answered it better?

**Run the cell. You should see** one tool call with `{"ward": "4B"}`, then an answer quoting real scores.

<details><summary>Hint</summary>

Both values come from things already in scope: the `call` object the service sent you, and the `result` your function returned. `json.dumps` turns a dict into a string.

</details>

<details><summary>Show solution code</summary>

```python
CALL_ID = call.call_id
OUTPUT_JSON = json.dumps(result)
```

</details>

In [ ]:
ANALYST_INSTRUCTIONS = (
    "You support the infection prevention and control team at one hospital. "
    "Use get_ipc_assessment for anything about how a ward is performing, and never "
    "estimate or invent a score. Quote the scores you retrieved. Be concise."
)

analyst = project.agents.create_version(
    agent_name=f"day2-ipc-analyst-{SUFFIX}",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=ANALYST_INSTRUCTIONS,
        tools=[assessment_tool],
    ),
)
AGENT_REF = {"agent_reference": {"type": "agent_reference", "name": analyst.name,
                                "version": str(analyst.version)}}
print(f"Created {analyst.name} version {analyst.version}\n")

LOCAL_TOOLS = {"get_ipc_assessment": get_ipc_assessment, "schedule_ipc_review": schedule_ipc_review}


def run(question, agent_ref=None, approved=frozenset(), max_rounds=5):
    """Ask a question and service every function call the agent makes."""
    agent_ref = agent_ref or AGENT_REF
    transcript = []
    response = client.responses.create(input=question, extra_body=agent_ref)

    for _ in range(max_rounds):
        calls = [item for item in response.output if item.type == "function_call"]
        transcript += [item.type for item in response.output if item.type != "reasoning"]
        if not calls:
            break

        outputs = []
        for call in calls:
            arguments = json.loads(call.arguments)  # arguments arrive as a JSON string
            if call.name in NEEDS_APPROVAL and call.name not in approved:
                result = {"status": "declined", "reason": "A human did not approve this action."}
                print(f"  BLOCKED  {call.name}({arguments}) - no approval")
            else:
                result = LOCAL_TOOLS[call.name](**arguments)
                print(f"  called   {call.name}({arguments})")

            CALL_ID = ...  # TODO 2: what ties this result back to the request.
            OUTPUT_JSON = ...  # TODO 2: the result, as a JSON string.
            check_todos(CALL_ID=CALL_ID, OUTPUT_JSON=OUTPUT_JSON)
            outputs.append({"type": "function_call_output", "call_id": CALL_ID, "output": OUTPUT_JSON})

        response = client.responses.create(
            previous_response_id=response.id, input=outputs, extra_body=agent_ref
        )

    return response, transcript


NEEDS_APPROVAL = set()  # you will revisit this in section 5

print("A QUESTION A DASHBOARD WOULD ANSWER BETTER\n")
simple, simple_transcript = run("How is Ward 4B doing on staffing and workload?")
print(f"\n  items: {' -> '.join(simple_transcript)}\n")
print(simple.output_text)

That worked, and it was overkill. One lookup, rendered as a sentence. A table would have been faster, cheaper and deterministic, and you should say so out loud when someone proposes an agent for this.

## 4. The question no dashboard answers

Now give the same agent the Lab 3 knowledge base alongside its function tool. Two tools, two systems, two owners:

| Tool | Runs where | Holds |
|---|---|---|
| `who_guidelines` | A remote MCP server | What WHO says every facility should do |
| `get_ipc_assessment` | This notebook | What one ward actually does |

Neither half is useful alone. The guideline does not know your hospital exists. Your hospital's scores do not explain what "good" means or what to do about a gap. **The answer lives in the join, and until now a human made that join by reading a 91-page PDF.**

Note what you are *not* doing: routing. You do not tell the agent which tool answers which part. You describe both tools well and let it decide - that is the declarative model from the top of this lab, and it is why the agent handles questions nobody designed a screen for.

Adding a tool is not free. Every tool definition costs input tokens on every request whether it is used or not, and a model choosing between many similar tools picks worse. Two well-described tools is a good place to be; forty is a design problem.

**Run the cell. You should see** an `mcp_call` and a `function_call` in the same run, then an answer that names a WHO recommendation *and* Ward 4B's score against it.

You may also see the *same* function called twice in one round. Models do issue parallel and sometimes redundant calls, which is why the loop iterates over every `function_call` in the output rather than assuming there is one. Returning a result for each is required: miss one `call_id` and the run stalls waiting for it.

In [ ]:
kb_tool = MCPTool(
    server_label="who_guidelines",
    server_url=KB_MCP_URL,
    project_connection_id=KB_CONNECTION_NAME,
    allowed_tools=["knowledge_base_retrieve"],
    require_approval="never",
)

advisor = project.agents.create_version(
    agent_name=f"day2-ipc-advisor-{SUFFIX}",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "You advise the infection prevention and control team at one hospital.\n"
            "Use who_guidelines for what WHO recommends, and cite the guideline.\n"
            "Use get_ipc_assessment for how a ward actually scores, and never invent a score.\n"
            "Answer by connecting the two: name the gap, say what WHO asks for, and "
            "recommend what to fix first. Be concise and do not give clinical advice."
        ),
        tools=[assessment_tool, kb_tool],
    ),
)
ADVISOR_REF = {"agent_reference": {"type": "agent_reference", "name": advisor.name,
                                  "version": str(advisor.version)}}
print(f"Created {advisor.name} version {advisor.version}, with 2 tools\n")

COMPOSITE_QUESTION = (
    "Ward 4B is failing its IPC audit. Which of its weakest areas does WHO treat as a core "
    "component, what does WHO actually require there, and what should we fix first?"
)
joined, joined_transcript = run(COMPOSITE_QUESTION, agent_ref=ADVISOR_REF)
print(f"\n  items: {' -> '.join(joined_transcript)}\n")
print(joined.output_text)

## 5. An action that changes something

Everything so far only read. Now the agent can book a review, and the calculus changes: a bad retrieval produces a bad sentence, while a bad action produces a meeting, an email, and someone's afternoon.

Here is the asymmetry to remember. For **MCP** tools, Foundry has a built-in approval mechanism - `require_approval="always"` makes the service pause and emit an approval request you respond to. For **function** tools there is no such thing, and there cannot be: the service never runs your code, so it has nothing to pause. **You are the runtime, so you are the gate.** It is an `if` in your loop, and if you do not write it, nothing else will.

That is why the loop above already has this line:

```python
if call.name in NEEDS_APPROVAL and call.name not in approved:
```

The rule of thumb: restrict function tools to safe operations, or require explicit confirmation for anything that changes data.

The cell runs the same request twice - once with approval withheld, once granted - and prints `SCHEDULED_REVIEWS` after each. In a real application `approved` comes from a person clicking a button, not from a variable.

### To-Do 3 - Gate the action

**Goal:** the agent cannot change data without a human.

**Steps**

1. Put the name of the tool that writes data into `NEEDS_APPROVAL`. Leave the read-only one out.
2. Run the cell and compare the two runs.

**Predict:** in the declined run, does the agent report that it booked the review?

**Run the cell. You should see** `BLOCKED` and an empty list in the first run, then a booking and one entry in the second.

<details><summary>Hint</summary>

A set of tool names. Only one of the two functions in `LOCAL_TOOLS` writes anything.

</details>

<details><summary>Show solution code</summary>

```python
NEEDS_APPROVAL = {"schedule_ipc_review"}
```

</details>

In [ ]:
NEEDS_APPROVAL = ...  # TODO 3: the names of tools a human must approve.
check_todos(NEEDS_APPROVAL=NEEDS_APPROVAL)

review_tool = FunctionTool(
    type="function",
    name="schedule_ipc_review",
    description=(
        "Book an internal IPC review meeting for one ward and one WHO core component. "
        "This creates a real booking, so only call it when the user asks for one."
    ),
    parameters={
        "type": "object",
        "properties": {
            "ward": {"type": "string", "description": "Ward identifier: 4B, 2A or ICU."},
            "component": {"type": "string", "description": "The WHO core component to review."},
            "reason": {"type": "string", "description": "One sentence on why the review is needed."},
        },
        "required": ["ward", "component", "reason"],
        "additionalProperties": False,
    },
    strict=True,
)

coordinator = project.agents.create_version(
    agent_name=f"day2-ipc-coordinator-{SUFFIX}",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "You coordinate infection prevention and control reviews at one hospital. "
            "Check a ward's assessment before booking anything. If a tool reports that an "
            "action was declined, say so plainly and do not claim it succeeded."
        ),
        tools=[assessment_tool, review_tool],
    ),
)
COORDINATOR_REF = {"agent_reference": {"type": "agent_reference", "name": coordinator.name,
                                      "version": str(coordinator.version)}}

REQUEST = "Ward 4B is weakest on workload and staffing. Please book a review of that component."

print("RUN 1 - APPROVAL WITHHELD")
declined, _ = run(REQUEST, agent_ref=COORDINATOR_REF, approved=frozenset())
print(f"\n{declined.output_text}\n  bookings now: {SCHEDULED_REVIEWS}\n")

print("RUN 2 - APPROVAL GRANTED")
allowed, _ = run(REQUEST, agent_ref=COORDINATOR_REF, approved=frozenset(NEEDS_APPROVAL))
print(f"\n{allowed.output_text}\n  bookings now: {SCHEDULED_REVIEWS}")

## Deterministic success check

Model wording varies, so this checks structure: the round trip completed, the composite question used both tools, and the approval gate actually stopped a write.

In [ ]:
assert simple.status == "completed", "The first run did not complete."
assert "function_call" in simple_transcript, (
    "No function call happened. Check the tool description and that tools=[assessment_tool] was set."
)
assert "function_call" in joined_transcript and "mcp_call" in joined_transcript, (
    f"The composite question used {joined_transcript}. It should use both the knowledge base "
    "and the local function."
)
assert "4B" in joined.output_text, "The joined answer never mentions the ward it was asked about."
assert len(SCHEDULED_REVIEWS) == 1, (
    f"Expected exactly one booking, found {len(SCHEDULED_REVIEWS)}. The withheld run should have "
    "created none and the approved run exactly one."
)
assert SCHEDULED_REVIEWS[0]["ward"] == "4B", "The booking went to the wrong ward."
assert "schedule_ipc_review" in NEEDS_APPROVAL, "The tool that writes data should need approval."
assert "get_ipc_assessment" not in NEEDS_APPROVAL, (
    "A read-only tool does not need an approval gate. Gating everything trains people to click yes."
)
print("PASS - the round trip completed, both tools answered one question, and the approval "
      f"gate held: {len(SCHEDULED_REVIEWS)} booking after two attempts.")

## What you learned

- A tool is a **description**. The model decides when to call it, so `description` is functional code, not documentation.
- **Who runs the code** is the distinction that matters. Service-run and MCP tools execute elsewhere; a function tool executes in your process, which is why it needs a round trip.
- The round trip: find `function_call` in `response.output`, parse `arguments` from JSON, execute, return a `function_call_output` carrying `call_id` and a **string** `output`, and continue with `previous_response_id`.
- `azure-ai-projects`' `FunctionTool` is a schema, not a callable. Nothing runs your function but you.
- Combining a knowledge tool with a system-of-record tool answers questions **neither one can**, and that join is where an agent beats a dashboard.
- For a plain lookup, a dashboard wins. Say so.
- Function tools have **no built-in approval**. You are the runtime, so the gate is your `if`. MCP tools have `require_approval` because the service runs those.
- Every attached tool costs input tokens on every request, used or not.

**Reflection.** One sentence each.

1. Your agent never calls `get_ipc_assessment` and answers from memory instead. What do you change first?
2. You gate every tool behind approval, including read-only ones. What goes wrong?
3. Ward 4B's score changes tomorrow. What has to be redeployed?

<details><summary>Compare your answers</summary>

1. The `description`, then the instructions. "No function call occurs" is nearly always a naming or description problem, not a model problem - the model never understood the tool applied. Making the description say *when* to use it, and that the data cannot be guessed, fixes most cases.
2. People stop reading them. An approval that is always granted is a click, not a control, and it hides the one request that mattered. Gate writes; leave reads alone.
3. Nothing. The agent holds no scores - it holds a description of how to fetch them. That is the same property the knowledge base gave you in Lab 3, and it is the main reason to put facts behind a tool rather than in a prompt.

</details>

**What this does not prove:** the agent chose the right tools here, but tool choice is not guaranteed. With many similar tools, selection accuracy falls, and nothing in this lab measures how often it picks correctly. That is an evaluation problem, which is Lab 7.

**If something fails,** this table covers nearly everything:

| Symptom | Likely cause | Fix |
|---|---|---|
| A function call, but no final answer | You never returned the output | Send `function_call_output` and continue with `previous_response_id` |
| No function call at all | Tool missing from the definition, or a weak description | Confirm `tools=[...]`, then sharpen the name and description |
| Arguments are not valid JSON | Schema mismatch | Check types, `required`, and `strict` |
| Wrong parameters | Ambiguous description | Describe each parameter, with examples |
| Outputs rejected as expired | The 10-minute run limit | Return promptly; poll slow work separately |
| `tool_limit_exceeded` | More than 128 tools | Remove unused tools |

**Reset:** the cleanup cell closes local clients only. Three agent versions remain in Foundry; delete `day2-ipc-analyst-...`, `day2-ipc-advisor-...` and `day2-ipc-coordinator-...` from the portal or with `project.agents.delete(...)`. The JSON file is unchanged - bookings only ever lived in memory.

**Further reading:** [function calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling), [the MCP tool](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol), [tool best practices](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/tool-best-practice), [what Toolbox is](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/toolbox-overview), and [limits and regional tool support](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/limits-quotas-regions).

**Expected artifact:** a run where an `mcp_call` and a `function_call` answer one question together, and an approval gate that blocked a write and then allowed it.

**Next:** Lab 7 measures whether any of this is actually any good.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed the local clients. Your three agents remain in Foundry.")